# N-gram Language Models

**ANLP Session 6 — Foundations of Language Modelling**

---

Before transformer-based models like GPT or BERT, language models were built on a deceptively simple idea: **count how often words appear together**.

This notebook builds an n-gram language model from scratch and uses it to generate text. The goal is not to produce a competitive model — it's to make the core language modelling objective concrete before we look at how neural models solve the same problem.

**What is a language model?**

A language model assigns a probability to a sequence of words:

$$P(w_1, w_2, \ldots, w_n)$$

Using the **chain rule of probability**, this factorises as:

$$P(w_1, w_2, \ldots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, w_2, \ldots, w_{i-1})$$

The challenge: conditioning on *all* previous words is intractable for any real corpus. N-gram models solve this with the **Markov assumption** — only the last $n-1$ words matter:

$$P(w_i \mid w_1, \ldots, w_{i-1}) \approx P(w_i \mid w_{i-(n-1)}, \ldots, w_{i-1})$$

| N-gram type | Condition on | Example |
|---|---|---|
| Unigram | nothing | $P(\text{cat})$ |
| Bigram | previous 1 word | $P(\text{sat} \mid \text{the})$ |
| Trigram | previous 2 words | $P(\text{mat} \mid \text{the, cat})$ |

**What you'll build:**
1. Tokenise and clean a text corpus
2. Count n-grams and convert counts to probabilities
3. Generate new text by sampling from the trigram distribution
4. Explore the limitations that motivated neural language models

---

## Setup

In [1]:
import re
import random
import math
from collections import defaultdict, Counter

random.seed(42)  # reproducible output
print("Ready.")

Ready.


---

## Part 1 — Corpus and Tokenisation

We use a small passage of public-domain text so every step is inspectable. The same code scales to any plain-text corpus — swap `CORPUS` for the contents of a file and the rest of the notebook works unchanged.

### 1.1 Tokenisation

For n-gram models we need a flat list of tokens. We use a simple whitespace-and-punctuation tokeniser:
- lowercase everything so `The` and `the` count as the same token
- insert `<START>` and `<END>` markers at sentence boundaries so the model learns which words begin and end sentences

In [2]:
CORPUS = """
The cat sat on the mat. The cat wore a hat. The hat was flat.
A big black cat sat on a small red mat near the door.
The dog ran into the garden and sat under the tree.
She picked up the hat and put it on the cat.
The cat did not like the hat on its head.
It jumped off the mat and ran into the garden.
The dog chased the cat around the garden and the cat jumped over the fence.
The sun was shining and the sky was blue and the garden was full of flowers.
A bird sat on the fence and sang a song about the sun and the sky.
The cat watched the bird with wide green eyes from behind the tree.
"""


def tokenise(text):
    """
    Split text into sentences, then tokens.

    Each sentence is wrapped with <START> and <END> markers so the model
    can learn sentence-initial and sentence-final distributions separately.
    """
    # Split on sentence-ending punctuation, keeping the delimiter
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    tokens = []
    for sentence in sentences:
        # Lowercase and split on non-alphabetic characters
        words = re.findall(r"[a-z']+", sentence.lower())
        if words:
            tokens += ["<START>"] + words + ["<END>"]
    return tokens


tokens = tokenise(CORPUS)

print(f"Total tokens : {len(tokens)}")
print(f"Unique tokens: {len(set(tokens))}")
print()
print("First 20 tokens:")
print(tokens[:20])

Total tokens : 154
Unique tokens: 59

First 20 tokens:
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>', '<START>', 'the', 'cat', 'wore', 'a', 'hat', '<END>', '<START>', 'the', 'hat', 'was', 'flat']


### 1.2 Vocabulary

The **vocabulary** is the set of all unique tokens in the corpus. Its size directly affects model complexity — every new word encountered at test time that wasn't in training is an **out-of-vocabulary (OOV)** token.

In [3]:
vocab = sorted(set(tokens))
word_freq = Counter(tokens)

print(f"Vocabulary size: {len(vocab)}")
print()
print("10 most common tokens:")
for word, count in word_freq.most_common(10):
    bar = "█" * count
    print(f"  {word:<12} {count:>3}  {bar}")

Vocabulary size: 59

10 most common tokens:
  the           28  ████████████████████████████
  <START>       12  ████████████
  <END>         12  ████████████
  cat            8  ████████
  and            8  ████████
  on             5  █████
  a              5  █████
  sat            4  ████
  hat            4  ████
  was            4  ████


---

## Part 2 — Counting N-grams

### 2.1 Counting

The raw material for any n-gram model is a frequency table. We scan the token list with a sliding window and increment a counter for each observed sequence.

In [4]:
unigram_counts = Counter(tokens)

bigram_counts = defaultdict(int)
trigram_counts = defaultdict(int)

for i in range(len(tokens) - 2):
    w1, w2, w3 = tokens[i], tokens[i + 1], tokens[i + 2]

    bigram_counts[(w1, w2)] += 1
    trigram_counts[(w1, w2, w3)] += 1

# The final pair only forms a bigram, not a trigram
bigram_counts[(tokens[-2], tokens[-1])] += 1

print(f"Unigrams : {len(unigram_counts):>5}")
print(f"Bigrams  : {len(bigram_counts):>5}")
print(f"Trigrams : {len(trigram_counts):>5}")
print()

# Inspect the most frequent trigrams
print("Top 10 trigrams by frequency:")
for ngram, count in sorted(trigram_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {' '.join(ngram):<30} {count}")

Unigrams :    59
Bigrams  :   106
Trigrams :   134

Top 10 trigrams by frequency:
  <END> <START> the              7
  <START> the cat                4
  cat sat on                     2
  sat on the                     2
  <END> <START> a                2
  <START> the dog                2
  ran into the                   2
  into the garden                2
  the garden and                 2
  the tree <END>                 2


### 2.2 Converting Counts to Probabilities

We estimate the probability of the next word using **Maximum Likelihood Estimation (MLE)**. For a trigram $(w_1, w_2, w_3)$:

$$P(w_3 \mid w_1, w_2) = \frac{\text{count}(w_1, w_2, w_3)}{\text{count}(w_1, w_2)}$$

This is the relative frequency of the trigram — how often we see $w_3$ following $w_1\ w_2$, divided by how often we see $w_1\ w_2$ at all.

In [5]:
# --- Unigram probabilities ---
total_tokens = sum(unigram_counts.values())
unigram_probs = {
    word: count / total_tokens
    for word, count in unigram_counts.items()
}

# --- Bigram probabilities ---
# P(w2 | w1) = count(w1, w2) / count(w1)
bigram_probs = {
    (w1, w2): count / unigram_counts[w1]
    for (w1, w2), count in bigram_counts.items()
}

# --- Trigram probabilities ---
# P(w3 | w1, w2) = count(w1, w2, w3) / count(w1, w2)
trigram_probs = {
    (w1, w2, w3): count / bigram_counts[(w1, w2)]
    for (w1, w2, w3), count in trigram_counts.items()
}

print("Sample conditional probabilities — P(w3 | w1, w2)")
print("-" * 55)
examples = [
    ("the", "cat"),
    ("the", "mat"),
    ("<START>", "the"),
    ("sat", "on"),
]

for w1, w2 in examples:
    # Find all trigrams that start with (w1, w2)
    continuations = {
        w3: prob
        for (a, b, w3), prob in trigram_probs.items()
        if a == w1 and b == w2
    }
    if continuations:
        top = sorted(continuations.items(), key=lambda x: -x[1])[:4]
        print(f"  P(? | '{w1}', '{w2}'):")
        for w3, p in top:
            bar = "█" * int(p * 20)
            print(f"    {w3:<15} {p:.3f}  {bar}")
    else:
        print(f"  P(? | '{w1}', '{w2}'): — no trigrams observed")
    print()

Sample conditional probabilities — P(w3 | w1, w2)
-------------------------------------------------------
  P(? | 'the', 'cat'):
    sat             0.143  ██
    wore            0.143  ██
    <END>           0.143  ██
    did             0.143  ██

  P(? | 'the', 'mat'):
    <END>           0.500  ██████████
    and             0.500  ██████████

  P(? | '<START>', 'the'):
    cat             0.500  ██████████
    dog             0.250  █████
    hat             0.125  ██
    sun             0.125  ██

  P(? | 'sat', 'on'):
    the             0.667  █████████████
    a               0.333  ██████



---

## Part 3 — Generating Text

### 3.1 The Generation Loop

Text generation with an n-gram model follows the same autoregressive pattern we'll see again with GPT:

1. Start with a context (the seed)
2. Look up the distribution over the next word given the current context
3. **Sample** one word from that distribution
4. Append it to the output and slide the context window forward
5. Repeat until `<END>` is generated or a maximum length is reached

Sampling (rather than always picking the most probable word) introduces variety — different runs produce different sentences even from the same seed.

In [32]:
def sample_next_word(context, n_probs, fallback_probs):
    """
    Sample the next word given a bigram context.

    Parameters
    ----------
    context : tuple[str, str]
        The two most recent tokens (w1, w2).
    n_probs : dict
        Trigram probability table {(w1, w2, w3): prob}.
    fallback_probs : dict
        Bigram probability table {(w1, w2): prob} — used when the trigram
        context was never seen in training (data sparsity fallback).

    Returns
    -------
    str : the sampled next token
    """
    w1, w2 = context

    # Collect all completions for this context
    completions = {
        w3: prob
        for (a, b, w3), prob in n_probs.items()
        if a == w1 and b == w2
    }

    if completions:
        words = list(completions.keys())
        weights = list(completions.values())
        return random.choices(words, weights=weights, k=1)[0]

    # Fallback: bigram model (ignore w1, condition only on w2)
    fallback = {
        w3: prob
        for (a, w3), prob in fallback_probs.items()
        if a == w2
    }
    if fallback:
        words = list(fallback.keys())
        weights = list(fallback.values())
        return random.choices(words, weights=weights, k=1)[0]

    # Last resort: uniform random word from vocabulary
    return random.choice(vocab)


def generate(seed=("<START>", "the"), max_tokens=25,
             trigram_p=trigram_probs, bigram_p=bigram_probs):
    """
    Generate a sentence by sampling from the trigram model.

    Parameters
    ----------
    seed : tuple[str, str]
        Starting context. Use ("<START>", word) to begin a new sentence.
    max_tokens : int
        Hard limit to prevent infinite loops on unseen contexts.

    Returns
    -------
    str : the generated sentence (without the <START>/<END> markers)
    """
    context = seed
    output = [seed[1]] if seed[0] == "<START>" else list(seed)

    for _ in range(max_tokens):
        next_word = sample_next_word(context, trigram_p, bigram_p)
        if next_word == "<END>":
            break
        if next_word == "<START>":
            break
        output.append(next_word)
        context = (context[1], next_word)  # slide the window

    return " ".join(output)


# Generate several sentences from different seeds
seeds = [
    ("<START>", "the"),
    ("<START>", "the"),
    ("<START>", "the"),
    ("<START>", "a"),
    ("the", "cat"),
    ("the", "dog"),
]

print("Generated sentences:")
print("-" * 60)
for seed in seeds:
    sentence = generate(seed=seed)
    # Capitalise the first word and add a full stop
    print(f"  {sentence.capitalize()}.")

Generated sentences:
------------------------------------------------------------
  The hat on its head.
  The hat and put it on the fence.
  The cat wore a hat.
  A big black cat sat on the mat and ran into the garden and the garden was full of flowers.
  The cat around the garden and the sky was blue and the sky was blue and the sky was blue and the sky was blue and the.
  The dog ran into the garden.


---

## Part 4 — Evaluating the Model: Perplexity

How do we measure how good a language model is? The standard metric is **perplexity**.

Intuitively, perplexity measures how *surprised* the model is by a test sentence. A lower perplexity means the model assigned higher probability to the observed words — it found the sentence predictable.

$$\text{Perplexity}(W) = P(w_1, w_2, \ldots, w_N)^{-1/N} = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid w_{i-2}, w_{i-1})\right)$$

A perplexity of $k$ means the model is as uncertain as if it had to choose uniformly among $k$ equally likely words at every step.

In [31]:
def sentence_perplexity(sentence, trigram_p, bigram_p, unigram_p):
    """
    Compute the trigram perplexity of a sentence.

    Falls back to bigram or unigram probability when higher-order
    n-gram counts are zero (simple back-off, no smoothing).
    """
    toks = tokenise(sentence)
    if len(toks) < 3:
        return float("inf")

    log_prob = 0.0
    n = 0

    for i in range(len(toks) - 2):
        w1, w2, w3 = toks[i], toks[i + 1], toks[i + 2]

        p = trigram_p.get((w1, w2, w3))

        if p is None:
            # Back off to bigram
            p = bigram_p.get((w2, w3))

        if p is None:
            # Back off to unigram
            p = unigram_p.get(w3, 1e-10)  # small floor for truly unseen words

        log_prob += math.log(p)
        n += 1

    return math.exp(-log_prob / n)


# Compare in-corpus vs out-of-corpus sentences
test_sentences = [
    # High probability — matches corpus patterns
    ("The cat sat on the mat.", "in-corpus pattern"),
    ("The dog ran into the garden.", "in-corpus pattern"),
    # Novel but plausible — similar words, new combinations
    ("The cat ran into the garden.", "novel combination"),
    ("A bird sat on the mat.", "novel combination"),
    # Unlikely — unusual word order / words
    ("The sky sat on the cat.", "unusual"),
    ("Flowers sang about the dog.", "unusual"),
]

print(f"{'Sentence':<45} {'Type':<22} {'Perplexity':>10}")
print("-" * 80)
for sentence, label in test_sentences:
    ppl = sentence_perplexity(sentence, trigram_probs, bigram_probs, unigram_probs)
    print(f"  {sentence:<43} {label:<22} {ppl:>10.1f}")

Sentence                                      Type                   Perplexity
--------------------------------------------------------------------------------
  The cat sat on the mat.                     in-corpus pattern             2.2
  The dog ran into the garden.                in-corpus pattern             1.8
  The cat ran into the garden.                novel combination             2.9
  A bird sat on the mat.                      novel combination             1.6
  The sky sat on the cat.                     unusual                       5.3
  Flowers sang about the dog.                 unusual                      21.2


---

## Part 5 — The Sparsity Problem and Smoothing

The biggest weakness of n-gram models is **data sparsity**: most n-grams that could conceivably occur will never appear in any finite corpus. This means the model assigns **zero probability** to many perfectly grammatical sentences — and zero probability breaks perplexity (log(0) is undefined).

**Laplace (add-1) smoothing** is the simplest fix: pretend every n-gram was observed at least once by adding 1 to every count.

$$P_{\text{Laplace}}(w_3 \mid w_1, w_2) = \frac{\text{count}(w_1, w_2, w_3) + 1}{\text{count}(w_1, w_2) + |V|}$$

where $|V|$ is the vocabulary size. This ensures no probability is ever exactly zero — at the cost of shifting some probability mass from observed to unobserved events.

In [28]:
V = len(vocab)  # vocabulary size

def laplace_trigram_prob(w1, w2, w3, trigram_counts, bigram_counts, V):
    """
    Add-1 smoothed trigram probability.

    P_laplace(w3 | w1, w2) = (count(w1,w2,w3) + 1) / (count(w1,w2) + |V|)
    """
    numerator   = trigram_counts.get((w1, w2, w3), 0) + 1
    denominator = bigram_counts.get((w1, w2), 0) + V
    return numerator / denominator


# Compare raw MLE vs Laplace for a seen and an unseen trigram
examples = [
    ("the", "cat", "sat"),    # observed in corpus
    ("the", "cat", "flew"),   # NOT observed ("flew" not in corpus)
    ("<START>", "the", "cat"),  # observed
    ("<START>", "the", "penguin"),  # completely unseen word
]

print(f"{'Trigram':<35} {'MLE':>10} {'Laplace':>10}")
print("-" * 58)

for w1, w2, w3 in examples:
    mle = trigram_probs.get((w1, w2, w3), 0.0)
    lap = laplace_trigram_prob(w1, w2, w3, trigram_counts, bigram_counts, V)
    label = f"('{w1}', '{w2}', '{w3}')"
    print(f"  {label:<33} {mle:>10.4f} {lap:>10.4f}")

print()
print("Laplace smoothing ensures no trigram has zero probability.")
print("The trade-off: probability mass is redistributed to unseen events,")
print("which can make common events appear less likely than they really are.")

Trigram                                    MLE    Laplace
----------------------------------------------------------
  ('the', 'cat', 'sat')                 0.1429     0.0303
  ('the', 'cat', 'flew')                0.0000     0.0152
  ('<START>', 'the', 'cat')             0.5000     0.0746
  ('<START>', 'the', 'penguin')         0.0000     0.0149

Laplace smoothing ensures no trigram has zero probability.
The trade-off: probability mass is redistributed to unseen events,
which can make common events appear less likely than they really are.


---

## Summary

| Concept | What we built | Key limitation |
|---|---|---|
| Unigram model | $P(w)$ from word frequency | Ignores all context |
| Bigram model | $P(w_i \mid w_{i-1})$ | Only 1 word of context |
| Trigram model | $P(w_i \mid w_{i-2}, w_{i-1})$ | Sparsity grows with $n$ |
| Laplace smoothing | Add-1 to all counts | Crude redistribution of mass |
| Perplexity | $\exp(-\frac{1}{N} \sum \log P)$ | Lower is better |

### Why neural language models?

N-gram models face a hard trade-off:
- **Small $n$**: fast and estimable, but misses long-range patterns
- **Large $n$**: better context, but exponential sparsity

Neural language models escape this trade-off by **representing words as dense vectors** (embeddings) rather than discrete symbols. Instead of counting every distinct sequence, the model *learns* a function that maps a variable-length context to a probability distribution — sharing statistical strength across similar contexts even if the exact sequence was never observed.

The **transformer architecture** (Session 6, notebook 01) takes this further: the attention mechanism allows it to condition on *any position in the context*, not just the last $n-1$ tokens.
